In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
%pwd

'c:\\Users\\roozb\\my_files\\University\\Boston University\\3-Fall 2026\\1-DX702 Experimental Design and Causality\\Homework'

In [3]:
import os
os.getcwd()

'c:\\Users\\roozb\\my_files\\University\\Boston University\\3-Fall 2026\\1-DX702 Experimental Design and Causality\\Homework'

In [4]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/roozb/my_files/University/Boston University/3-Fall 2026/1-DX702 Experimental Design and Causality/Homework')

Task:

Given data about an instrumental variable, find the effect. X is the treatment, W the confounder, Y the outcome, and Z the instrument. Use homework_4.1.csv. 


## Question 1
As in the explanation, try averaging the Y difference and X difference (over W and Z) in two ways: 

Subtract the average Y value for Z equals 1 and Z equals 0. Subtract the average X value for Z equals 1 and Z equals 0. Divide the two.   
Find the average Y value for Z equals 1 and Z equals 0 for a narrow range of W. Find the average X value for Z equals 1 and Z equals 0 for the same narrow range of W. Take the ratio to find the effect. Then average this over all the ranges of W.


For the first item, the effect is closest to: 

- A: 1

- B: 0.5 

- C: 2

- D: 1.5

In [5]:
df1 = pd.read_csv('./data/homework_4.1.csv')
df1.head()

,Z,W,X,Y
0,0,-0.155644,-0.496971,0.282484
1,1,0.529539,2.284240,4.740596
2,1,0.910514,0.872232,3.449569
3,1,-0.705476,2.157260,3.002531
4,0,-0.590874,-0.386730,-1.848796


In [6]:
(df1.loc[df1["Z"] ==1, "Y"].mean() - df1.loc[df1["Z"] ==0, "Y"].mean())/(df1.loc[df1["Z"] ==1, "X"].mean() - df1.loc[df1["Z"] ==0, "X"].mean())

np.float64(1.5618587073765746)

In [8]:
bins = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
res_n = {}
for n in bins:
    df_n = df1.copy()
    df_n["W"] = pd.cut(df_n["W"], bins=n, labels=False)
    res = []
    for cat in df_n["W"].unique():
        numi = df_n.loc[((df_n["Z"] == 1) & (df_n["W"] == cat)), "Y"].mean() - df_n.loc[((df_n["Z"] == 0) & (df_n["W"] == cat)), "Y"].mean()
        deni = df_n.loc[((df_n["Z"] == 1) & (df_n["W"] == cat)), "X"].mean() - df_n.loc[((df_n["Z"] == 0) & (df_n["W"] == cat)), "X"].mean()
        res.append(numi/deni)

    res = np.array(res)
    weights = df_n["W"].value_counts(normalize=True).values
    mask = ~np.isnan(res)
    res_n[n] = np.average(res[mask], weights=weights[mask])

In [12]:
np.mean(list(res_n.values()))

np.float64(1.473562762237333)

Answer : D
___

## Question 2

Did you need to know W to do this? 

- A: No 

- B: Yes 

Answer : A 
____

Task: 

Given student data involving test scores (X), a cutoff, and an outcome (Y), which measures whether the students got into college (as in the example in the text), determine whether the math course helps students get into college in each dataset. Use datasets homework_4.2.a and homework_4.2.b. 

## Question 3

Given a cutoff score of 80, which dataset seems most likely to involve a nonzero linear term, allowing Y to relate linearly to X before and after the cutoff? 

- A: Dataset b (X2, Y2)

- B: Dataset a (X, Y)

In [9]:
df2a = pd.read_csv('./data/homework_4.2.a.csv')
df2b = pd.read_csv('./data/homework_4.2.b.csv')
display(df2a.head())
display(df2b.head())

,X,Y
0,81.822339,1
1,92.487870,0
2,85.372460,0
3,78.828025,0
4,75.807080,1


,X2,Y2
0,76.643034,1
1,87.743397,1
2,81.639469,1
3,73.740485,0
4,90.480268,1


In [10]:
cutoff = 80

df2a["centered_X"] = df2a["X"] - cutoff
df2a["treated"] = (df2a["X"] >= cutoff).astype(int)
df2a["interaction"] = df2a["centered_X"] * df2a["treated"]

df2b["centered_X2"] = df2b["X2"] - cutoff
df2b["treated"] = (df2b["X2"] >= cutoff).astype(int) 
df2b["interaction2"] = df2b["centered_X2"] * df2b["treated"]

display(df2a.head())
display(df2b.head())

,X,Y,centered_X,treated,interaction
0,81.822339,1,1.822339,1,1.822339
1,92.487870,0,12.487870,1,12.487870
2,85.372460,0,5.372460,1,5.372460
3,78.828025,0,-1.171975,0,-0.000000
4,75.807080,1,-4.192920,0,-0.000000


,X2,Y2,centered_X2,treated,interaction2
0,76.643034,1,-3.356966,0,-0.000000
1,87.743397,1,7.743397,1,7.743397
2,81.639469,1,1.639469,1,1.639469
3,73.740485,0,-6.259515,0,-0.000000
4,90.480268,1,10.480268,1,10.480268


In [11]:
X_a = df2a[["centered_X", "treated", "interaction"]]
X_a = sm.add_constant(X_a)

model_a = sm.OLS(df2a["Y"], X_a).fit()

print(model_a.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.090
Model:                            OLS   Adj. R-squared:                  0.090
Method:                 Least Squares   F-statistic:                     3300.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):               0.00
Time:                        13:30:07   Log-Likelihood:                -67412.
No. Observations:              100000   AIC:                         1.348e+05
Df Residuals:                   99996   BIC:                         1.349e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.3050      0.004     86.481      

In [12]:
X_b = df2b[["centered_X2", "treated", "interaction2"]]
X_b = sm.add_constant(X_b)

model_b = sm.OLS(df2b["Y2"], X_b).fit()

print(model_b.summary())

                            OLS Regression Results                            
Dep. Variable:                     Y2   R-squared:                       0.160
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     6360.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):               0.00
Time:                        13:30:07   Log-Likelihood:                -45123.
No. Observations:              100000   AIC:                         9.025e+04
Df Residuals:                   99996   BIC:                         9.029e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.7023      0.003    249.432   

Answer: A  
___

## Question 4

In that dataset, is Y increasing or decreasing before the cutoff? 

- A: Decreasing 

- B: Increasing

Answer: B

## Question 5

Is Y's slope higher or lower after the cutoff compared with before? 

- A: Lower 

- B: Higher 

Answer: A